# Point 3 — Exploration des Données
## Deep Learning for Crop Classification Using Multi-Source Satellite Data
### M1 SII — USTHB — 2025/2026

Ce notebook effectue l'analyse exploratoire des données collectées depuis Google Earth Engine :
- Distribution des classes
- Visualisation des séries temporelles NDVI (comme la Figure 2 de l'article)
- Analyse des patterns temporels par culture
- Inspection des valeurs manquantes
- Statistiques spectrales

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

print('Librairies chargées avec succès')

## 1. Chargement des données

⚠️ **Modifie le chemin ci-dessous** pour pointer vers tes fichiers CSV.

In [ ]:
# ========== MODIFIER CES CHEMINS ==========
ARK_PATH = 'Arkansas_10k.csv'
CAL_PATH = 'California_10k.csv'
# ===========================================

ark = pd.read_csv(ARK_PATH)
cal = pd.read_csv(CAL_PATH)

print(f'Arkansas : {ark.shape[0]} lignes × {ark.shape[1]} colonnes')
print(f'California : {cal.shape[0]} lignes × {cal.shape[1]} colonnes')

## 2. Mapping des classes et organisation des bandes

Les bandes sont nommées automatiquement par GEE : B2, B2_1, B2_2... (B2 = timestep 0, B2_1 = timestep 1, etc.).
On les réorganise en matrice (10 bandes × 36 timesteps) pour chaque pixel.

In [ ]:
# Mapping des classes
ARK_CLASSES = {0: 'Corn', 1: 'Cotton', 2: 'Rice', 3: 'Soybeans', 4: 'Others'}
CAL_CLASSES = {0: 'Rice', 1: 'Alfalfa', 2: 'Grapes', 3: 'Almonds', 4: 'Pistachios', 5: 'Others'}

# Couleurs pour chaque culture (cohérent avec l'article)
ARK_COLORS = {0: '#2ecc71', 1: '#e74c3c', 2: '#3498db', 3: '#f39c12', 4: '#95a5a6'}
CAL_COLORS = {0: '#3498db', 1: '#f39c12', 2: '#2ecc71', 3: '#1a5276', 4: '#e74c3c', 5: '#95a5a6'}

# Les 10 bandes spectrales Sentinel-2
BAND_NAMES = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
N_BANDS = 10
N_TIMESTEPS = 36

# DOY (Day of Year) pour chaque timestep
DOY = [i * 10 + 5 for i in range(N_TIMESTEPS)]  # milieu de chaque intervalle de 10 jours

print(f'Bandes: {BAND_NAMES}')
print(f'Timesteps: {N_TIMESTEPS}')
print(f'DOY: {DOY[:5]}... {DOY[-3:]}')

In [ ]:
def get_band_columns(df, band_name):
    """
    Récupère les 36 colonnes d'une bande dans l'ordre temporel.
    GEE nomme: B8 (t=0), B8_1 (t=1), B8_2 (t=2), ..., B8_35 (t=35)
    """
    cols = []
    for t in range(N_TIMESTEPS):
        if t == 0:
            col_name = band_name
        else:
            col_name = f'{band_name}_{t}'
        if col_name in df.columns:
            cols.append(col_name)
        else:
            cols.append(None)
    return cols


def extract_band_timeseries(df, band_name):
    """
    Extrait la série temporelle d'une bande pour tous les pixels.
    Retourne un array (n_samples, 36)
    """
    cols = get_band_columns(df, band_name)
    valid_cols = [c for c in cols if c is not None]
    return df[valid_cols].values


def compute_ndvi(df):
    """
    Calcule le NDVI = (NIR - Red) / (NIR + Red) = (B8 - B4) / (B8 + B4)
    pour chaque pixel et chaque timestep.
    Retourne un array (n_samples, 36)
    """
    nir = extract_band_timeseries(df, 'B8')  # NIR
    red = extract_band_timeseries(df, 'B4')  # Red
    
    # Éviter la division par zéro
    denominator = nir + red
    ndvi = np.where(denominator > 0, (nir - red) / denominator, 0)
    
    return ndvi


# Test
print('Colonnes B8 (5 premières):', get_band_columns(ark, 'B8')[:5])
print('Shape NDVI Arkansas:', compute_ndvi(ark).shape)

## 3. Distribution des classes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Arkansas
ark_counts = ark['label'].value_counts().sort_index()
bars1 = axes[0].bar(
    [ARK_CLASSES[i] for i in ark_counts.index.astype(int)],
    ark_counts.values,
    color=[ARK_COLORS[i] for i in ark_counts.index.astype(int)],
    edgecolor='white', linewidth=0.5
)
for bar, val in zip(bars1, ark_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val}\n({val/len(ark)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Arkansas — Distribution des classes', fontweight='bold')
axes[0].set_ylabel('Nombre de pixels')
axes[0].set_ylim(0, max(ark_counts.values) * 1.2)

# California
cal_counts = cal['label'].value_counts().sort_index()
bars2 = axes[1].bar(
    [CAL_CLASSES[i] for i in cal_counts.index.astype(int)],
    cal_counts.values,
    color=[CAL_COLORS[i] for i in cal_counts.index.astype(int)],
    edgecolor='white', linewidth=0.5
)
for bar, val in zip(bars2, cal_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val}\n({val/len(cal)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[1].set_title('California — Distribution des classes', fontweight='bold')
axes[1].set_ylabel('Nombre de pixels')
axes[1].set_ylim(0, max(cal_counts.values) * 1.2)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : class_distribution.png')

## 4. Courbes NDVI par culture (reproduction de la Figure 2 de l'article)

C'est la visualisation principale de l'article : la moyenne du NDVI pour chaque culture au cours de l'année.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# --- ARKANSAS ---
ndvi_ark = compute_ndvi(ark)
labels_ark = ark['label'].values

for cls_id, cls_name in ARK_CLASSES.items():
    mask = labels_ark == cls_id
    if mask.sum() == 0:
        continue
    mean_ndvi = ndvi_ark[mask].mean(axis=0)
    axes[0].plot(DOY, mean_ndvi, '-o', label=cls_name, color=ARK_COLORS[cls_id],
                markersize=4, linewidth=2)

axes[0].set_title('(a) Arkansas', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Day of Year')
axes[0].set_ylabel('Mean NDVI Value')
axes[0].set_xlim(0, 370)
axes[0].set_ylim(-0.1, 1.0)
axes[0].legend(loc='upper left', ncol=3)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.3)

# --- CALIFORNIA ---
ndvi_cal = compute_ndvi(cal)
labels_cal = cal['label'].values

for cls_id, cls_name in CAL_CLASSES.items():
    mask = labels_cal == cls_id
    if mask.sum() == 0:
        continue
    mean_ndvi = ndvi_cal[mask].mean(axis=0)
    axes[1].plot(DOY, mean_ndvi, '-o', label=cls_name, color=CAL_COLORS[cls_id],
                markersize=4, linewidth=2)

axes[1].set_title('(b) California', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Day of Year')
axes[1].set_ylabel('Mean NDVI Value')
axes[1].set_xlim(0, 370)
axes[1].set_ylim(-0.1, 1.0)
axes[1].legend(loc='upper left', ncol=3)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('ndvi_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : ndvi_timeseries.png')

## 5. Analyse des valeurs manquantes

Les valeurs à 0 représentent les données manquantes (nuages, absence d'image).
Le module ALPE du MCTNet est conçu pour gérer ces valeurs.

In [ ]:
def compute_missing_rate_per_timestep(df):
    """
    Calcule le taux de zéros par pas de temps.
    Un pixel est considéré manquant à un timestep si TOUTES ses 10 bandes sont à 0.
    """
    rates = []
    for t in range(N_TIMESTEPS):
        cols = []
        for band in BAND_NAMES:
            col = band if t == 0 else f'{band}_{t}'
            if col in df.columns:
                cols.append(col)
        if cols:
            # Un pixel est manquant si toutes les bandes sont à 0
            all_zero = (df[cols] == 0).all(axis=1).mean() * 100
            rates.append(all_zero)
        else:
            rates.append(100)
    return rates


ark_missing = compute_missing_rate_per_timestep(ark)
cal_missing = compute_missing_rate_per_timestep(cal)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Arkansas
colors_ark = ['#e74c3c' if r > 80 else '#f39c12' if r > 50 else '#3498db' for r in ark_missing]
axes[0].bar(DOY, ark_missing, width=8, color=colors_ark, edgecolor='white', linewidth=0.5)
axes[0].set_title('Arkansas — Taux de données manquantes par pas de temps', fontweight='bold')
axes[0].set_xlabel('Day of Year')
axes[0].set_ylabel('% pixels manquants')
axes[0].set_ylim(0, 105)
axes[0].axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='50%')
axes[0].legend()

# California
colors_cal = ['#e74c3c' if r > 80 else '#f39c12' if r > 50 else '#3498db' for r in cal_missing]
axes[1].bar(DOY, cal_missing, width=8, color=colors_cal, edgecolor='white', linewidth=0.5)
axes[1].set_title('California — Taux de données manquantes par pas de temps', fontweight='bold')
axes[1].set_xlabel('Day of Year')
axes[1].set_ylabel('% pixels manquants')
axes[1].set_ylim(0, 105)
axes[1].axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='50%')
axes[1].legend()

plt.tight_layout()
plt.savefig('missing_data.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : missing_data.png')

## 6. Distribution du taux de données manquantes par échantillon

L'article (Figure 15) analyse le taux de données manquantes par échantillon.
On reproduit cette analyse ici.

In [ ]:
def compute_missing_rate_per_sample(df):
    """
    Pour chaque pixel, calcule le % de timesteps manquants (toutes bandes à 0).
    """
    missing_per_sample = []
    for t in range(N_TIMESTEPS):
        cols = []
        for band in BAND_NAMES:
            col = band if t == 0 else f'{band}_{t}'
            if col in df.columns:
                cols.append(col)
        if cols:
            is_missing = (df[cols] == 0).all(axis=1).astype(int)
            missing_per_sample.append(is_missing)
    
    missing_matrix = np.column_stack(missing_per_sample)
    return missing_matrix.mean(axis=1) * 100  # % manquant par échantillon


ark_sample_missing = compute_missing_rate_per_sample(ark)
cal_sample_missing = compute_missing_rate_per_sample(cal)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ark_sample_missing, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Arkansas — Taux manquant par échantillon', fontweight='bold')
axes[0].set_xlabel('% timesteps manquants')
axes[0].set_ylabel('Nombre de pixels')
axes[0].axvline(x=np.mean(ark_sample_missing), color='red', linestyle='--',
               label=f'Moyenne: {np.mean(ark_sample_missing):.1f}%')
axes[0].legend()

axes[1].hist(cal_sample_missing, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].set_title('California — Taux manquant par échantillon', fontweight='bold')
axes[1].set_xlabel('% timesteps manquants')
axes[1].set_ylabel('Nombre de pixels')
axes[1].axvline(x=np.mean(cal_sample_missing), color='red', linestyle='--',
               label=f'Moyenne: {np.mean(cal_sample_missing):.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('missing_per_sample.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : missing_per_sample.png')

## 7. Profils spectraux moyens par culture

Visualisation des signatures spectrales moyennes (10 bandes) pendant la saison de croissance.

In [ ]:
def get_peak_season_spectrum(df, labels, classes, t_start=15, t_end=25):
    """
    Calcule le spectre moyen pendant la saison de croissance (DOY 150-250)
    pour chaque culture.
    """
    spectra = {}
    for cls_id, cls_name in classes.items():
        mask = labels == cls_id
        if mask.sum() == 0:
            continue
        band_means = []
        for band in BAND_NAMES:
            vals = []
            for t in range(t_start, t_end):
                col = band if t == 0 else f'{band}_{t}'
                if col in df.columns:
                    v = df.loc[mask, col].values
                    v = v[v > 0]  # exclure les manquants
                    if len(v) > 0:
                        vals.extend(v)
            band_means.append(np.mean(vals) if vals else 0)
        spectra[cls_name] = band_means
    return spectra


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Arkansas
spectra_ark = get_peak_season_spectrum(ark, ark['label'].values, ARK_CLASSES)
for cls_name, spectrum in spectra_ark.items():
    cls_id = [k for k, v in ARK_CLASSES.items() if v == cls_name][0]
    axes[0].plot(BAND_NAMES, spectrum, '-o', label=cls_name, color=ARK_COLORS[cls_id],
                markersize=5, linewidth=2)
axes[0].set_title('Arkansas — Profil spectral (DOY 150-250)', fontweight='bold')
axes[0].set_xlabel('Bande spectrale')
axes[0].set_ylabel('Réflectance moyenne')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# California
spectra_cal = get_peak_season_spectrum(cal, cal['label'].values, CAL_CLASSES)
for cls_name, spectrum in spectra_cal.items():
    cls_id = [k for k, v in CAL_CLASSES.items() if v == cls_name][0]
    axes[1].plot(BAND_NAMES, spectrum, '-o', label=cls_name, color=CAL_COLORS[cls_id],
                markersize=5, linewidth=2)
axes[1].set_title('California — Profil spectral (DOY 150-250)', fontweight='bold')
axes[1].set_xlabel('Bande spectrale')
axes[1].set_ylabel('Réflectance moyenne')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('spectral_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : spectral_profiles.png')

## 8. Heatmap des corrélations entre bandes

Visualisation de la corrélation entre les 10 bandes spectrales pendant la saison de croissance.

In [ ]:
def get_peak_season_data(df, t_start=18, t_end=22):
    """
    Extrait les données des 10 bandes pour quelques timesteps de la saison de croissance.
    """
    data = {}
    for band in BAND_NAMES:
        vals = []
        for t in range(t_start, t_end):
            col = band if t == 0 else f'{band}_{t}'
            if col in df.columns:
                vals.append(df[col].values)
        data[band] = np.concatenate(vals) if vals else np.zeros(len(df))
    return pd.DataFrame(data)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))

peak_ark = get_peak_season_data(ark)
peak_cal = get_peak_season_data(cal)

sns.heatmap(peak_ark.corr(), annot=True, fmt='.2f', cmap='RdYlBu_r',
           ax=axes[0], vmin=-1, vmax=1, square=True, cbar_kws={'shrink': 0.8})
axes[0].set_title('Arkansas — Corrélation des bandes', fontweight='bold')

sns.heatmap(peak_cal.corr(), annot=True, fmt='.2f', cmap='RdYlBu_r',
           ax=axes[1], vmin=-1, vmax=1, square=True, cbar_kws={'shrink': 0.8})
axes[1].set_title('California — Corrélation des bandes', fontweight='bold')

plt.tight_layout()
plt.savefig('band_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée : band_correlation.png')

## 9. Statistiques récapitulatives

In [ ]:
print('=' * 60)
print('RÉCAPITULATIF')
print('=' * 60)

for name, df, classes in [('Arkansas', ark, ARK_CLASSES), ('California', cal, CAL_CLASSES)]:
    spectral = [c for c in df.columns if c != 'label']
    print(f'\n--- {name} ---')
    print(f'  Points: {len(df)}')
    print(f'  Bandes spectrales: {len(spectral)}')
    print(f'  Classes: {len(classes)}')
    print(f'  NaN: {df.isnull().sum().sum()}')
    print(f'  Taux global de zéros: {(df[spectral] == 0).sum().sum() / (len(df) * len(spectral)) * 100:.1f}%')
    print(f'  Min: {df[spectral].min().min():.1f}')
    print(f'  Max: {df[spectral].max().max():.1f}')
    print(f'  Moyenne: {df[spectral].mean().mean():.1f}')
    print(f'\n  Distribution des classes:')
    for cls_id, cls_name in classes.items():
        count = (df['label'] == cls_id).sum()
        pct = count / len(df) * 100
        print(f'    {cls_id} ({cls_name:12s}): {count:5d} ({pct:5.1f}%)')

print('\n' + '=' * 60)
print('Exploration terminée. Prochaine étape : Prétraitement (Point 4)')
print('=' * 60)